# Question 1.2 — Key Assumptions of IEThresh

**Paper:** Efficiently Learning the Accuracy of Labeling Sources for Selective Sampling — Donmez, Carbonell, Schneider (KDD 2009)

**Student:** Yashi Gupta (Roll No. 230072)

## Assumption 1: Each Labeler Has Accuracy Greater Than 0.5 (Better Than Random)

**Assumption:** The paper assumes that every individual labeler has a labeling accuracy strictly greater than 0.5 for binary classification tasks. That is, each labeler is more likely to assign the correct label than the incorrect one for any given instance.

**Why the Method Needs It:** IEThresh relies on majority vote among selected oracles to approximate the true label (Equation 3). The reward function compares each oracle's label against this majority vote — an oracle is rewarded with 1 if it agrees with the majority and 0 otherwise. If labelers can be worse than random (accuracy < 0.5), the majority vote could systematically reflect the WRONG label, because the majority of oracles would converge on the incorrect answer. This would cause the reward signal to be inverted: the worst oracles (those consistently wrong) would agree with the wrong majority vote and receive the highest rewards, while the few accurate oracles would be penalized for disagreeing. The entire confidence-interval mechanism would then preferentially select the bad oracles.

**Violation Scenario:** On Amazon Mechanical Turk, some workers may intentionally provide wrong answers (spammers who click randomly or adversarially) or may have fundamentally misunderstood the task instructions. In the TEMP dataset used in the paper's own experiments, some annotators had accuracy as low as 0.44, which the authors themselves acknowledge violates this assumption. In such cases, if enough low-quality workers are present, the majority vote becomes unreliable and the entire reward estimation framework breaks down.

**Paper Reference:** Section 3.2 — "we assume an individual labeler accuracy is better than random guess, i.e. > 0.5 in the binary case." Also acknowledged in Section 4.1 regarding the AMT datasets where some annotators fell below this threshold.

## Assumption 2: Labeler Accuracy Is Stationary Over Time

**Assumption:** The paper assumes that each oracle's labeling accuracy remains constant across all iterations of the active learning process. An oracle that starts with 80% accuracy maintains that same 80% accuracy from the first query to the last.

**Why the Method Needs It:** The Interval Estimation framework (Equation 1) computes confidence intervals using the sample mean and standard deviation of ALL past rewards from an oracle, treating them as i.i.d. draws from a fixed distribution. If an oracle's accuracy changes over time (e.g., due to fatigue, learning, or shifting attention), the historical rewards no longer reflect the oracle's current reliability. Early rewards would contaminate the estimate of current performance. For instance, if an oracle started strong but deteriorated, the algorithm would continue to over-trust it because the high early rewards inflate the mean. Conversely, an oracle that improved over time would remain penalized by its poor early performance. The t-distribution confidence interval in Equation 1 is only statistically valid under the stationarity assumption.

**Violation Scenario:** In a medical annotation task where domain experts annotate pathology images over several hours, annotator fatigue would cause accuracy to decline over time. Similarly, a crowd-worker on Mechanical Turk might start carefully but become careless after annotating hundreds of instances. The opposite could also happen — an annotator might improve as they get familiar with the task. In any of these cases, the running average reward would lag behind the oracle's true current performance, and the confidence intervals would not correctly represent the oracle's present reliability.

**Paper Reference:** Section 5 (Conclusions) — the authors explicitly identify this as a limitation and future direction: "One major direction is to track variable oracle performance over time since it could change depending on numerous reasons, e.g. oracle fatigue."

## Assumption 3: Labeling Errors Are Independent Across Oracles

**Assumption:** The paper assumes that when multiple oracles label the same instance, their errors are statistically independent. If Oracle A makes a mistake on instance x, this does not make it more or less likely that Oracle B will also make a mistake on the same instance. Each oracle's error is drawn independently based on its own accuracy rate.

**Why the Method Needs It:** The majority vote's reliability as an approximation of the true label fundamentally depends on error independence. Under the Condorcet jury theorem logic, if each labeler has >0.5 accuracy and errors are independent, the probability that a majority of labelers are simultaneously wrong decreases exponentially as more labelers are included. IEThresh leverages this by using majority vote as a proxy for ground truth in the reward computation (Equation 3). If errors are correlated — for example, all labelers tend to fail on the same ambiguous instances — then majority vote can be systematically wrong on precisely those hard instances. Since the reward signal is derived from majority vote, correlated errors would corrupt the oracle quality estimates in a way that the confidence intervals cannot detect.

**Violation Scenario:** In a sentiment classification task, sarcastic sentences are inherently ambiguous and most annotators would misclassify them in the same way. If several oracles share the same training background, cultural context, or cognitive biases, their errors would be highly correlated — they would all fail on the same types of instances, and majority vote would confidently assign the wrong label. Another example: if annotators can see each other's labels (as in some collaborative annotation platforms), their responses become dependent, violating independence.

**Paper Reference:** Section 5 (Conclusions) — "Another direction is to relax the assumption that the noise generation is uncorrelated. It is possible that the labelers make correlated errors as noted by [15]." The reference [15] points to Sheng et al.'s work on repeated labeling.